# Run a simulation

**Model:** `pfc_models.models.logpfc_extv`  
**Written against:** pfc-models 0.0.0  
**Takeaway:** how to build the external-potential model, step it, and view the density field

Migrated 2026-10-12 from `log/gpu-spectral-extV-implementation/_testSimulation.ipynb`. Code cells are carried across unchanged apart from import rewrites; stored outputs were stripped, so re-run to regenerate the figures.

> **Deferred features.** This notebook runs the reduced `logpfc_extv`: spectral Laplacian, ETD1 stepper. The finite-difference Laplacian and the predictor-corrector step used when these results were first produced are deferred for a later refactor, so re-running will not reproduce the original output.

> **Seeding.** The hot initial-condition generators now draw from `np.random.default_rng(seed)` instead of seeding global NumPy state. Call sites are unchanged, but a given integer seed produces a different noise field than it did in the source fork.

In [ ]:
# numerics
import numpy as np
import cupy as cp

# plotting
import plotly.io as pio

# Set the global default renderer
pio.renderers.default = "png"

from importlib import reload

In [ ]:
# density plotting
import pfc_models.viz.heatmap as _densityHeatmap
reload(_densityHeatmap)
from pfc_models.viz.heatmap import *

In [ ]:
# Initial conditions
import pfc_models.fields.initial_conditions_2d as _sim_initial_conditions
reload(_sim_initial_conditions)
from pfc_models.fields.initial_conditions_2d import *

In [ ]:
# Simulation
import pfc_models.models.logpfc_extv.simulation as Simulation
reload(Simulation)
from pfc_models.models.logpfc_extv.simulation import *

# Simulation Builder

In [ ]:
# NOTE on the migrated model: this is the REDUCED logpfc_extv - spectral
# Laplacian, ETD1 stepper. The finite-difference Laplacian symbol and the
# predictor-corrector step that the original gpu-spectral-extV fork used are
# deferred for a later refactor and are not present here; see model.md for both
# original expressions. Output below will not match the original fork's.

# geometry for verifiable derivatives
Nx = 250
Ny = 250
a = 4*np.pi/3**0.5
h = 4*np.pi
Mx = 28
My = 16    
Lx = Mx * a
Ly = My * h

# model parameters
Gamma_val = 1.0
beta_val = 0.5
q0 = 1.0
epsilon_val = -0.9
g_val = -2.0
meanDensity_S = 0.55
meanDensity_V = 0.3
relStd = 0.1
kMax = np.sqrt(3) / 2
V0 = -1e-4
potWidth = 10.0

geometry = BuildGeometry(Nx=Nx, Ny=Ny, Lx=Lx, Ly=Ly)
modelParameters = BuildLogPFCModel_withVacancyForce_Parameters(
  Gamma=Gamma_val,
  beta=beta_val,
  q0=q0,
  epsilon=epsilon_val,
  g=g_val,
  dt=7.5e-3,
  meanDensity=meanDensity_S,
  V0=V0,
  potWidth=potWidth
)

vacancyCoordsSet = [(7,4), (21,4)]

sim = BuildCupyGPU_Simulation(modelParameters, geometry)
phiInit = GeneratePerfectCrystalInitialWithVancies(geometry, meanDensity_S, meanDensity_V, vacancyCoordsSet, A0=0.15)
# phiInit = GenerateHotInitialConditions(geometry, meanDensity, relStd, kMax)
sim.State.phi[...] = cp.asarray(phiInit, dtype=cpFloatType)
print(
    geometry.X.shape,
    geometry.KX.shape,
    geometry.X[0,1], geometry.Y[1,0],
  )

X, Y = sim.Geometry.X, sim.Geometry.Y
GetDensityHeatmap(sim.phi.get(), X, Y, title='ϕ', showScale=True).show(renderer="notebook")

# Test Simulation

In [ ]:
# # For testing purposes only:
# sim.V[[0],...] = cp.asarray(np.cos(X), dtype=cpFloatType)
# sim.V[[1],...] = cp.asarray(np.cos(X)*np.cos(Y), dtype=cpFloatType)

In [ ]:
# # Do batch1: N_μ = log(ϕ) + gϕ^2 + ϕ^3 and phi, V in k-space
# sim._Batch1_CalcNonlinMu()
# GetDensityHeatmap(sim.State.nonlin_mu.get(), X, Y, title='Nonlinear μ', showScale=True).show()

In [ ]:
# # Do batch2: μ, ⟨μ⟩
# sim._Batch2_CalcMu_Terms()
# GetDensityHeatmap(sim.mu.get(), X, Y, title='μ', showScale=True).show()
# GetDensityHeatmap(sim.State.local_mu.get(), X, Y, title='⟨μ⟩', showScale=True).show()
# GetDensityHeatmap(sim.State.grad_mu[0,...].get(), X, Y, title='∇μ (x)', showScale=True).show()
# GetDensityHeatmap(sim.State.grad_mu[1,...].get(), X, Y, title='∇μ (y)', showScale=True).show()

In [ ]:
# # Do batch3: ∇̇⋅(ϕV), N_ϕ = Γ∇^2 N_μ - ϕV(t + ΔT)
# sim._Batch3_CalcPhiNext()
# GetDensityHeatmap(sim.State.phi_next.get(), X, Y, title='ϕ(t + ΔT)', showScale=True).show()

In [ ]:
totalT = 50.0
steps = 10
miniSteps = int(totalT / steps / sim.Model.dt)
print(f'Running {steps} steps of {miniSteps} mini-steps each (totalT = {totalT})')
for i in range(steps):
  print(f'Step {i}: ϕ.min = {sim.phi.get().min()}, ϕ.max = {sim.phi.get().max()}, ϕ.mean = {sim.phi.get().mean()}')
  for _ in range(miniSteps):
    sim.TimeStep()
print(f'Step {i+1}: ϕ.min = {sim.phi.get().min()}, ϕ.max = {sim.phi.get().max()}, ϕ.mean = {sim.phi.get().mean()}')

GetDensityHeatmap(sim.State.phi.get(), X, Y, title='ϕ(t + ΔT)', showScale=True).show()

In [ ]:
sim.mu.get().std()

In [ ]:
GetDensityHeatmap(sim.State.phi.get(), X, Y, title='ϕ', ppx=4, showScale=True).show(renderer="notebook")

In [ ]:
GetDensityHeatmap(sim.Get_local_phi(), X, Y, title='⟨ϕ⟩', ppx=4, showScale=True).show(renderer="png")

In [ ]:
sim._vacancyTracker.AddVacancy(sim.Get_local_phi(), 63, 64)
sim._vacancyTracker.AddVacancy(sim.Get_local_phi(), 187, 64)
sim._vacancyTracker.vacancyTrajectories

In [ ]:
fig = GetDensityHeatmap(sim.Get_local_phi(), X, Y, title='⟨ϕ⟩', ppx=4, showScale=True)
for i, _vacancyTrajectory in enumerate(sim._vacancyTracker.vacancyTrajectories):
    x_coords, y_coords = zip(*_vacancyTrajectory)
    fig.add_trace(
        go.Scatter(
            x=x_coords,
            y=y_coords,
            mode='lines+markers',
            name=f'Vacancy Trajectory {i}',
            line=dict(color='red', width=1),
            marker=dict(size=3, color='red', symbol='x')
        )
    )

fig.update_layout(
    showlegend=True,
    legend=dict(
        x=0.98,              # Places it near the right edge of the plot
        y=0.98,              # Places it near the top edge of the plot
        xanchor='right',     # Anchors the right edge of the legend box to the x-coordinate
        yanchor='top',       # Anchors the top edge of the legend box to the y-coordinate
        xref='paper',
        yref='paper',
        bgcolor='rgba(255, 255, 255, 0.7)', # Optional: Semi-transparent white background 
        bordercolor='rgba(0, 0, 0, 0.1)',    # Optional: Soft border line
        borderwidth=1
    )
)

fig.show(renderer="png")

In [ ]:
sim._externalV.SetPotential_alongX(
  0.1,
  0.5,
  sim._vacancyTracker.vacancyCoords[0][0],
  sim._vacancyTracker.vacancyCoords[1][0]
)

In [ ]:
x_axis = X[0,:]
y_index = int(Nx//2)
V = sim._externalV.V.get()[y_index,:]
go.Figure(
  data = [
    go.Scatter(x = x_axis, y = V)
  ]
)

In [ ]:
totalT = 1000.0 #τ
steps = 10
miniSteps = int(totalT / steps / sim.Model.dt)
print(f'Running {steps} steps of {miniSteps} mini-steps each (totalT = {totalT})')
for i in range(steps):
  print(f'Step {i}: ϕ.min = {sim.phi.get().min()}, ϕ.max = {sim.phi.get().max()}, ϕ.mean = {sim.phi.get().mean()}')
  for _ in range(miniSteps):
    sim.TimeStep()
print(f'Step {i+1}: ϕ.min = {sim.phi.get().min()}, ϕ.max = {sim.phi.get().max()}, ϕ.mean = {sim.phi.get().mean()}')

GetDensityHeatmap(sim.State.phi.get(), X, Y, title='ϕ(t)', showScale=True).show()

In [ ]:
GetDensityHeatmap(sim.State.phi.get(), X, Y, title='ϕ', ppx=4, showScale=True).show(renderer="notebook")

In [ ]:

fig = GetDensityHeatmap(sim.Get_local_phi(), X, Y, title='⟨ϕ⟩', colorscale='Turbo', ppx=4, showScale=True)
for i, _vacancyTrajectory in enumerate(sim._vacancyTracker.vacancyTrajectories):
    x_coords, y_coords = zip(*_vacancyTrajectory)
    fig.add_trace(
        go.Scatter(
            x=x_coords,
            y=y_coords,
            mode='lines+markers',
            name=f'Vacancy Trajectory {i}',
            line=dict(color='red', width=1),
            marker=dict(size=3, color='red', symbol='x')
        )
    )

fig.update_layout(
    showlegend=True,
    legend=dict(
        x=0.98,              # Places it near the right edge of the plot
        y=0.98,              # Places it near the top edge of the plot
        xanchor='right',     # Anchors the right edge of the legend box to the x-coordinate
        yanchor='top',       # Anchors the top edge of the legend box to the y-coordinate
        xref='paper',
        yref='paper',
        bgcolor='rgba(255, 255, 255, 0.7)', # Optional: Semi-transparent white background 
        bordercolor='rgba(0, 0, 0, 0.1)',    # Optional: Soft border line
        borderwidth=1
    )
)

fig.show(renderer="png")

In [ ]:
( np.asarray(sim._vacancyTracker.vacancyTrajectories[0])[:,0].min(),
  np.asarray(sim._vacancyTracker.vacancyTrajectories[0])[:,0].max(),
)

In [ ]:
local_phi = sim.Get_local_phi()
x_axis = X[0,:]
y_index = int(Nx//2)

go.Figure(
  
  data=[
    go.Scatter(x=x_axis, y=1 - local_phi[y_index,:], mode='lines', name='⟨ϕ⟩ (y = Ny/2)'),
  ],
  layout=go.Layout(
    title='Local density profile along y = Ny/2',
    xaxis=dict(title='x'),
    yaxis=dict(title='⟨ϕ⟩'),
  )
).show(renderer="png")